## Code of Conduct Deep Dive

In this notebook, we’ll explore how to define, manage, and enforce Code of Conduct (CoC) policies using the Enkrypt AI SDK.

These policies help enforce real-world compliance and behavior standards — especially useful for regulated industries like finance, healthcare, and legal services.

### What You’ll Learn

- What CoC policies are and why they matter
- How to atomize a policy from text or PDF
- How to create, modify, and delete CoC policies
- How to use CoC policies in live detection
- How to interpret detection outcomes using CoC logic

### Use Case Example: Mortgage Chatbot for a Financial Institution

We’ll work with a mortgage compliance policy to simulate a chatbot scenario. This policy will ensure the model does not give unauthorized financial advice or violate internal risk rules.

In [1]:

import os
from dotenv import load_dotenv
from enkryptai_sdk import CoCClient

load_dotenv()

ENKRYPTAI_API_KEY = os.getenv("ENKRYPTAI_API_KEY")

coc_client = CoCClient(api_key=ENKRYPTAI_API_KEY)

print("✅ CoC Client initialized.")

✅ CoC Client initialized.


### Working with Multiple Policies

In this notebook, we’ll walk through creating and managing **two separate Code of Conduct policies** — one for healthcare and one for mortgage lending.

This mirrors real-world use cases where an organization may operate across multiple compliance domains, each with distinct behavioral constraints.

### Files We'll Use

To keep things concrete, we’ll start with two sample policy documents:

- **`Healthcare Policy.pdf`** – contains behavioral guidelines for an AI assistant providing health-related information
- **`Mortgage Policy.pdf`** – outlines acceptable behavior for a chatbot providing home loan guidance

We’ll atomize both files, create named policies from them, and then explore how to modify, list, and apply these policies.

In [2]:
from datetime import datetime

# Generate a timestamped suffix for unique policy names
timestamp = datetime.now().strftime("%Y%m%d-%H%M")

# Correct file paths
healthcare_policy_path = "healthcare_policy.pdf"
mortgage_policy_path = "mortgage_policy.pdf"

# Timestamped policy names
healthcare_policy_name = f"healthcare-guidelines-policy-{timestamp}"
mortgage_policy_name = f"mortgage-guidelines-policy-{timestamp}"

print("📄 Ready to atomize:")
print("  - Healthcare:", healthcare_policy_path, "→", healthcare_policy_name)
print("  - Mortgage:", mortgage_policy_path, "→", mortgage_policy_name)

📄 Ready to atomize:
  - Healthcare: healthcare_policy.pdf → healthcare-guidelines-policy-20250508-2041
  - Mortgage: mortgage_policy.pdf → mortgage-guidelines-policy-20250508-2041


### Atomize the Policy Documents

The first step is to atomize each PDF. This process extracts individual rules from a larger policy document, so they can be enforced during detection.

We’ll atomize both the healthcare and mortgage policy files and inspect how many rules were extracted from each.


In [3]:
from enkryptai_sdk import GuardrailsClient

guardrails_client = GuardrailsClient(api_key=ENKRYPTAI_API_KEY)

# Atomize the healthcare policy
healthcare_atomize_response = guardrails_client.atomize_policy(file=healthcare_policy_path)

# Atomize the mortgage policy
mortgage_atomize_response = guardrails_client.atomize_policy(file=mortgage_policy_path)

# Confirm both succeeded
assert healthcare_atomize_response.is_successful(), "Healthcare policy atomization failed"
assert mortgage_atomize_response.is_successful(), "Mortgage policy atomization failed"

print("✅ Atomization complete.")
print("  - Healthcare Rules:", healthcare_atomize_response.total_rules)
print("  - Mortgage Rules:", mortgage_atomize_response.total_rules)

File name: healthcare_policy.pdf
Reading file: /home/ubuntu/enkryptai-sdk/usage/healthcare_policy.pdf
File name: mortgage_policy.pdf
Reading file: /home/ubuntu/enkryptai-sdk/usage/mortgage_policy.pdf
✅ Atomization complete.
  - Healthcare Rules: 26
  - Mortgage Rules: 18


### Preview Extracted Rules

Let’s print a few rules from each policy to get a sense of what was extracted during atomization.

In [4]:
# Print the first 3 rules from each policy
print("Healthcare Policy — First 3 Rules:")
for rule in healthcare_atomize_response.get_rules_list()[:3]:
    print("-", rule)

print("\nMortgage Policy — First 3 Rules:")
for rule in mortgage_atomize_response.get_rules_list()[:3]:
    print("-", rule)

Healthcare Policy — First 3 Rules:
- 1. The assistant's response must not be relied upon as the sole basis for clinical decisions when providing healthcare information.
- 2. Users must not request professional medical judgment from the AI assistant when seeking healthcare advice.
- 3. The assistant's response must not provide personalized medical diagnoses, treatments, or advice when asked about health-related issues.

Mortgage Policy — First 3 Rules:
- 1. The assistant's response must not provide inaccurate or misleading information about loan terms, APR, or fees.
- 2. The assistant's response must not provide loan offers without disclosing essential terms like interest rate, payment schedule, or total loan cost.
- 3. The assistant's response must not promote or suggest undisclosed kickbacks, referral fees, or conflicts of interest in settlement services.


### Save the Atomized Policies

Once a policy has been atomized, you can save it to your Enkrypt account under a unique name. This allows you to reuse it across guardrails detection, deployments, and red teaming tasks.

We’ll now save both the healthcare and mortgage policies using their extracted rules and timestamped names.

In [5]:
# Save the healthcare policy
add_healthcare_policy_response = coc_client.add_policy(
    policy_file=healthcare_policy_path,
    policy_name=healthcare_policy_name,
    policy_rules=healthcare_atomize_response.get_rules_list(),
    total_rules=healthcare_atomize_response.total_rules
)

# Save the mortgage policy
add_mortgage_policy_response = coc_client.add_policy(
    policy_file=mortgage_policy_path,
    policy_name=mortgage_policy_name,
    policy_rules=mortgage_atomize_response.get_rules_list(),
    total_rules=mortgage_atomize_response.total_rules
)

# Confirm success
print("✅ Policy Saving Results:")
print("  - Healthcare:", add_healthcare_policy_response.message)
print("  - Mortgage:", add_mortgage_policy_response.message)

assert add_healthcare_policy_response.message == "Policy details added successfully"
assert add_mortgage_policy_response.message == "Policy details added successfully"

✅ Policy Saving Results:
  - Healthcare: Policy details added successfully
  - Mortgage: Policy details added successfully


### List All Code of Conduct Policies

You can retrieve a full list of saved Code of Conduct policies using the SDK. This is useful for confirming that your policies were saved correctly, or for viewing existing policies before updating or deleting them.

In [24]:
# Get the list of all saved CoC policies
policies_list = coc_client.get_policy_list()

# Display policies in a formatted table
print("📋 3 Selected Code of Conduct Policies:")
print("-" * 80)
print(f"{'Policy Name':<50} | {'Total Rules':<15}")
print("-" * 80)

for policy in policies_list.policies[:3]:
    print(f"{policy.name:<50} | {policy.total_rules:<15}")
print("-" * 80)
print(f"Total Policies: {len(policies_list.policies)}")

# Print the structure of a policy object to understand available fields
print("\n📊 Policy Object Structure:")
if policies_list.policies:
    example_policy = policies_list.policies[0]
    print(f"Available fields in a policy object:")
    for attr in dir(example_policy):
        if not attr.startswith('_'):  # Skip private attributes
            value = getattr(example_policy, attr)
            if not callable(value):  # Skip methods
                print(f"  - {attr}: {type(value).__name__}")
else:
    print("No policies available to show structure")
    


📋 3 Selected Code of Conduct Policies:
--------------------------------------------------------------------------------
Policy Name                                        | Total Rules    
--------------------------------------------------------------------------------
BS - policy - 1                                    | 40             
Sample Airline Policy                              | 4              
Example Financial Enterprise Policy                | 29             
--------------------------------------------------------------------------------
Total Policies: 24

📊 Policy Object Structure:
Available fields in a policy object:
  - created_at: str
  - name: str
  - pdf_name: str
  - policy_id: int
  - policy_rules: str
  - project_name: str
  - total_rules: int
  - updated_at: str


### Modify an Existing Code of Conduct Policy

Sometimes you may want to update a policy by adding, removing, or editing rules. The SDK allows you to do this easily by providing a new rule list for a saved policy name.

Let’s add a new rule to our healthcare policy.

In [28]:
# Get the existing rules from the healthcare policy
existing_policy_details = coc_client.get_policy(policy_name=healthcare_policy_name)
existing_rules = existing_policy_details.get_rules_list()

existing_rules

["1. The assistant's response must not be relied upon as the sole basis for clinical decisions when providing healthcare information.",
 '2. Users must not request professional medical judgment from the AI assistant when seeking healthcare advice.',
 "3. The assistant's response must not provide personalized medical diagnoses, treatments, or advice when asked about health-related issues.",
 "4. The assistant's response should not replace human consultation, particularly in emergency or complex clinical situations.",
 '5. User queries should not request personalized medical diagnoses, treatments, or advice.',
 '6. Users must not rely solely on AI outputs for clinical decision-making without consulting a qualified healthcare professional.',
 "7. The assistant's response must not offer definitive medical diagnoses when asked about health conditions.",
 "8. The assistant's response must not provide treatment plans or medication prescriptions when users inquire about treatment options.",
 "

In [27]:

# Define a new rule to add
new_rule = "The assistant must never suggest medical treatments without disclaiming that it is not a certified medical provider."

# Add the new rule to the list
updated_rules = existing_rules + [new_rule]
updated_total = len(updated_rules)

updated_total

27

In [31]:

# Modify the policy using the updated rule list
modify_response = coc_client.modify_policy(
    policy_name=healthcare_policy_name,
    policy_file=healthcare_policy_path,
    policy_rules=updated_rules,
    total_rules=updated_total
)

print("✅ Modify Status:", modify_response.message)
assert modify_response.message == "Policy details updated successfully"

✅ Modify Status: Policy details updated successfully


### Remove a Rule from a Code of Conduct Policy

To delete a rule from a policy, you simply fetch the current rules, remove the one you no longer want, and re-save the policy using the modified rule list.

Let’s remove a rule from the mortgage policy.

In [32]:
# Get existing rules from the mortgage policy
mortgage_policy_details = coc_client.get_policy(policy_name=mortgage_policy_name)
rules = mortgage_policy_details.get_rules_list()

# View the first 3 rules for reference
print("First 3 Rules Before Deletion:")
for idx, rule in enumerate(rules[:3]):
    print(f"{idx}. {rule}")

# Remove a rule — for example, the first rule
rule_to_remove = rules[0]
updated_rules = [r for r in rules if r != rule_to_remove]
updated_total = len(updated_rules)

# Modify the policy with the updated rule list
modify_response = coc_client.modify_policy(
    policy_name=mortgage_policy_name,
    policy_file=mortgage_policy_path,
    policy_rules=updated_rules,
    total_rules=updated_total
)

print("\n🗑️ Removed Rule:", rule_to_remove)
print("✅ Modify Status:", modify_response.message)
assert modify_response.message == "Policy details updated successfully"

First 3 Rules Before Deletion:
0. 1. The assistant's response must not provide inaccurate or misleading information about loan terms, APR, or fees.
1. 2. The assistant's response must not provide loan offers without disclosing essential terms like interest rate, payment schedule, or total loan cost.
2. 3. The assistant's response must not promote or suggest undisclosed kickbacks, referral fees, or conflicts of interest in settlement services.

🗑️ Removed Rule: 1. The assistant's response must not provide inaccurate or misleading information about loan terms, APR, or fees.
✅ Modify Status: Policy details updated successfully


## 🎉 Great Job — You Now Have Two Active Code of Conduct Policies

You’ve successfully:

- Atomized two real-world policy documents (healthcare and mortgage)
- Saved them to your Enkrypt account with timestamped names
- Modified one to add a new rule
- Updated another by removing a rule

These policies are now ready to be used in runtime detections, red teaming evaluations, or deployment guardrails.

Next up, we’ll apply these policies to actual model inputs and see how they enforce safety and compliance in real-time.